# 3-Model Ensemble Data Extraction and Aggregation at Station Locations
## Reference Period: 1995–2014 | SSP2-4.5

**Purpose:**  
Extract the basin-specific 3-model ensemble precipitation from the Jordan-wide unified NetCDF file at each of the 49 observed station locations, then aggregate to the same temporal scales produced for single models in Notebook 01:

1. **Daily** time series (mm/day) — 1995–2014  
2. **Monthly totals** (mm/month) — calendar-year aggregation  
3. **Long-term monthly means** (mm/month) — 12-month climatology averaged    over 1995–2014

**Input NC file used:** `Jordan_3model_ensemble_ssp245_1995_2014.nc`  
(the 1995–2014 period slice from Notebook 05, which already contains the basin-specific ensemble for every grid cell)

**Station-to-grid matching:**  
The same nearest-grid-point lookup used in Notebook 01 is applied (k-d tree on the 0.1° grid), ensuring exact consistency between single-model and ensemble extractions at each station location.

**Outputs saved to:**  
`C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\ensemble data\`

```
ensemble data/
  daily/
    daily_ensemble_all_stations.csv
  monthly/
    monthly_ensemble_all_stations.csv
  long_term_monthly_mean/
    ltmm_ensemble_all_stations.csv
```


## 1. Import Libraries

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from pathlib import Path
from scipy.spatial import cKDTree
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

print("Libraries loaded.")
for lib, mod in [("numpy", np), ("pandas", pd), ("xarray", xr)]:
    print(f"  {lib:<12}: {mod.__version__}")


Libraries loaded.
  numpy       : 2.1.3
  pandas      : 2.2.3
  xarray      : 2025.12.0


## 2. Configuration

In [2]:
# ── Input paths ───────────────────────────────────────────────────────────────
STATION_MAPPING_FILE = Path(
    r"D:\RICAAR\Pr.New.Stations.Selection\OBSERVATIONS"
    r"\Station_Basin_Mapping.xlsx"
)

# Unified ensemble NC for the 1995-2014 period (from Notebook 05)
ENSEMBLE_NC = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\3 ensemble models\Jordan_3model_ensemble_ssp245_1995_2014.nc"
)

# ── Output paths ──────────────────────────────────────────────────────────────
OUTPUT_ROOT = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\ensemble data"
)
DAILY_OUT   = OUTPUT_ROOT / "daily"
MONTHLY_OUT = OUTPUT_ROOT / "monthly"
LTMM_OUT    = OUTPUT_ROOT / "long_term_monthly_mean"

for d in [DAILY_OUT, MONTHLY_OUT, LTMM_OUT]:
    d.mkdir(parents=True, exist_ok=True)

# ── Period ────────────────────────────────────────────────────────────────────
PERIOD_START = 1995
PERIOD_END   = 2014

MONTH_NAMES = {
    1:"January", 2:"February", 3:"March",    4:"April",
    5:"May",     6:"June",     7:"July",      8:"August",
    9:"September",10:"October",11:"November",12:"December"
}

PR_VAR = "prAdjust"

print("Configuration loaded.")
print(f"  Ensemble NC  : {ENSEMBLE_NC}")
print(f"  Output root  : {OUTPUT_ROOT}")
print(f"  Period       : {PERIOD_START}–{PERIOD_END}")


Configuration loaded.
  Ensemble NC  : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\3 ensemble models\Jordan_3model_ensemble_ssp245_1995_2014.nc
  Output root  : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\ensemble data
  Period       : 1995–2014


## 3. Load Station Locations

The same 49 selected stations used throughout this workflow are loaded from `Station_Basin_Mapping.xlsx`.


In [3]:
stations = pd.read_excel(STATION_MAPPING_FILE)
stations["Station_ID"]   = stations["Station_ID"].astype(str).str.strip()
stations["Station_Name"] = stations["Station_Name"].astype(str).str.strip()
stations["Basin"]        = stations["Basin"].astype(str).str.strip()

print(f"Stations loaded : {len(stations)}")
print(f"Unique basins   : {stations['Basin'].nunique()}")
print()
print(stations[["Station_ID","Station_Name","Basin",
                "Latitude","Longitude"]].to_string(index=False))


Stations loaded : 49
Unique basins   : 12

Station_ID                        Station_Name                 Basin  Latitude  Longitude
    AB0004                      KH.EL-WAHADNEH               N.R.S.W 32.328239  35.643545
    AD0002                              HARTHA      YARMOUK (JORDAN) 32.692571  35.844729
    AD0005                             UM QEIS      YARMOUK (JORDAN) 32.654533  35.679241
    AD0008                              KHARJA      YARMOUK (JORDAN) 32.658084  35.887122
    AD0011                         EN NUEIYIME      YARMOUK (JORDAN) 32.417193  35.911881
    AD0023                     JABER MUGHAYYIR      YARMOUK (JORDAN) 32.509924  36.199947
    AD0032     BAQURA MET.STATION (METEO DEPT) JORDAN VALLY (JORDAN) 32.612437  35.596982
    AE0003                           KAFR YUBA               N.R.S.W 32.543115  35.798953
    AE0004                           KAFR ASAD               N.R.S.W 32.600307  35.710913
    AH0002                       WADI EL-YABIS JORDAN VAL

## 4. Open Ensemble NetCDF and Match Stations to Grid

The Jordan-wide ensemble NC file is opened and the grid coordinates are extracted. A k-d tree is built on the flattened 2-D grid — identical to the approach used in Notebook 01 — to find the nearest grid cell centre for each of the 49 stations.


In [4]:
ds_ens = xr.open_dataset(ENSEMBLE_NC)

lats   = ds_ens["lat"].values    # (85,)
lons   = ds_ens["lon"].values    # (75,)
times  = pd.to_datetime(ds_ens["time"].values)
n_lat  = len(lats)
n_lon  = len(lons)

print(f"Ensemble NC opened.")
print(f"  Grid      : {n_lat} lat × {n_lon} lon")
print(f"  Time      : {times[0].date()} → {times[-1].date()}  ({len(times):,} steps)")
print(f"  Variables : {list(ds_ens.data_vars)}")
print()

# ── Build k-d tree on flattened (lat, lon) grid ───────────────────────────────
lon_grid, lat_grid = np.meshgrid(lons, lats)          # (n_lat, n_lon)
grid_coords = np.column_stack([lat_grid.ravel(), lon_grid.ravel()])
lat_idx_flat, lon_idx_flat = np.unravel_index(np.arange(len(grid_coords)),
                                               lat_grid.shape)
tree = cKDTree(grid_coords)

# ── Match every station ───────────────────────────────────────────────────────
DISTANCE_THRESHOLD_KM = 25.0

qc_rows = []
for _, stn in stations.iterrows():
    dist_deg, idx = tree.query([stn["Latitude"], stn["Longitude"]])
    li = int(lat_idx_flat[idx])
    lj = int(lon_idx_flat[idx])
    dist_km = dist_deg * 111.32
    qc_rows.append({
        "Station_ID"  : stn["Station_ID"],
        "Station_Name": stn["Station_Name"],
        "Basin"       : stn["Basin"],
        "Stn_Lat"     : round(stn["Latitude"],  6),
        "Stn_Lon"     : round(stn["Longitude"], 6),
        "Grid_Lat"    : round(lats[li], 4),
        "Grid_Lon"    : round(lons[lj], 4),
        "Lat_Idx"     : li,
        "Lon_Idx"     : lj,
        "Distance_km" : round(dist_km, 2),
        "Flag"        : "OK" if dist_km <= DISTANCE_THRESHOLD_KM else "DISTANT",
    })

qc_df = pd.DataFrame(qc_rows)
n_flagged = (qc_df["Flag"] == "DISTANT").sum()

print(f"Station-grid matching complete.")
print(f"  All matched  : {len(qc_df)}")
print(f"  Flagged (>{DISTANCE_THRESHOLD_KM} km) : {n_flagged}")
print()
print(qc_df[["Station_ID","Station_Name","Basin",
             "Grid_Lat","Grid_Lon","Distance_km","Flag"]].to_string(index=False))


Ensemble NC opened.
  Grid      : 85 lat × 75 lon
  Time      : 1995-01-01 → 2014-12-31  (7,305 steps)
  Variables : ['prAdjust', 'basin_id']

Station-grid matching complete.
  All matched  : 49
  Flagged (>25.0 km) : 0

Station_ID                        Station_Name                 Basin  Grid_Lat  Grid_Lon  Distance_km Flag
    AB0004                      KH.EL-WAHADNEH               N.R.S.W     32.35     35.65         2.53   OK
    AD0002                              HARTHA      YARMOUK (JORDAN)     32.65     35.85         4.78   OK
    AD0005                             UM QEIS      YARMOUK (JORDAN)     32.65     35.65         3.29   OK
    AD0008                              KHARJA      YARMOUK (JORDAN)     32.65     35.85         4.23   OK
    AD0011                         EN NUEIYIME      YARMOUK (JORDAN)     32.45     35.95         5.60   OK
    AD0023                     JABER MUGHAYYIR      YARMOUK (JORDAN)     32.55     36.15         7.13   OK
    AD0032     BAQURA MET.STAT

## 5. Extract Daily Ensemble Precipitation at Station Locations

For each of the 49 stations, the daily precipitation time series is extracted from the ensemble NC file at the matched grid cell. Because the NC already contains the basin-specific 3-model ensemble mean, no further averaging is needed — each extracted value is already the ensemble result for that location.

Output: `daily/daily_ensemble_all_stations.csv`  
Columns: `Date`, `Station_ID`, `Station_Name`, `Basin`, `pr_mm_day`


In [5]:
daily_csv = DAILY_OUT / "daily_ensemble_all_stations.csv"

if daily_csv.exists():
    print(f"[SKIP] Daily CSV already exists: {daily_csv}")
    daily_df = pd.read_csv(daily_csv, parse_dates=["Date"])
else:
    # Load the full precipitation array into memory once (already sliced to
    # 1995-2014 in the NC file, so this is only ~7305 × 85 × 75 × 4 bytes ≈ 177 MB)
    pr_array = ds_ens[PR_VAR].values    # (n_time, n_lat, n_lon), float32

    rows = []
    for _, stn in tqdm(qc_df.iterrows(), total=len(qc_df), desc="Extracting stations"):
        li = int(stn["Lat_Idx"])
        lj = int(stn["Lon_Idx"])
        ts = pr_array[:, li, lj].astype(float)   # (n_time,)

        for date, val in zip(times, ts):
            rows.append({
                "Date"        : date.date(),
                "Station_ID"  : stn["Station_ID"],
                "Station_Name": stn["Station_Name"],
                "Basin"       : stn["Basin"],
                "pr_mm_day"   : round(float(val), 4) if np.isfinite(val) else np.nan,
            })

    daily_df = pd.DataFrame(rows)
    daily_df["Date"] = pd.to_datetime(daily_df["Date"])
    daily_df.sort_values(["Station_ID","Date"], inplace=True)
    daily_df.reset_index(drop=True, inplace=True)
    daily_df.to_csv(daily_csv, index=False)
    print(f"Daily CSV saved: {daily_csv}")

print(f"  Rows : {len(daily_df):,}")
print(f"  Cols : {daily_df.columns.tolist()}")
print()
print("Preview:")
print(daily_df.head(6).to_string(index=False))


Extracting stations:   0%|          | 0/49 [00:00<?, ?it/s]

Daily CSV saved: C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\ensemble data\daily\daily_ensemble_all_stations.csv
  Rows : 357,945
  Cols : ['Date', 'Station_ID', 'Station_Name', 'Basin', 'pr_mm_day']

Preview:
      Date Station_ID   Station_Name   Basin  pr_mm_day
1995-01-01     AB0004 KH.EL-WAHADNEH N.R.S.W     4.6361
1995-01-02     AB0004 KH.EL-WAHADNEH N.R.S.W     3.6113
1995-01-03     AB0004 KH.EL-WAHADNEH N.R.S.W     2.2458
1995-01-04     AB0004 KH.EL-WAHADNEH N.R.S.W     2.1845
1995-01-05     AB0004 KH.EL-WAHADNEH N.R.S.W     0.0015
1995-01-06     AB0004 KH.EL-WAHADNEH N.R.S.W     0.0011


## 6. Aggregate Daily to Monthly Totals

Daily values are summed within each calendar month for each station

$$\text{MS}(y, m) = \sum_{d=1}^{n} P(y, m, d)$$

Output: `monthly/monthly_ensemble_all_stations.csv`  
Columns: `Station_ID`, `Station_Name`, `Basin`, `Year`, `Month`, `pr_mm_month`


In [6]:
monthly_csv = MONTHLY_OUT / "monthly_ensemble_all_stations.csv"

if monthly_csv.exists():
    print(f"[SKIP] Monthly CSV already exists: {monthly_csv}")
    monthly_df = pd.read_csv(monthly_csv)
else:
    daily_df["Year"]  = daily_df["Date"].dt.year
    daily_df["Month"] = daily_df["Date"].dt.month

    monthly_df = (
        daily_df
        .groupby(["Station_ID","Station_Name","Basin","Year","Month"], sort=True)
        ["pr_mm_day"]
        .sum()
        .reset_index()
        .rename(columns={"pr_mm_day": "pr_mm_month"})
    )
    monthly_df["pr_mm_month"] = monthly_df["pr_mm_month"].round(4)
    monthly_df.to_csv(monthly_csv, index=False)
    print(f"Monthly CSV saved: {monthly_csv}")

print(f"  Rows : {len(monthly_df):,}")
print()
print("Preview — first station, all months of 1995:")
sid0 = monthly_df["Station_ID"].iloc[0]
print(monthly_df[(monthly_df["Station_ID"] == sid0) &
                 (monthly_df["Year"] == PERIOD_START)]
      [["Station_ID","Station_Name","Basin","Year","Month","pr_mm_month"]]
      .to_string(index=False))


Monthly CSV saved: C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\ensemble data\monthly\monthly_ensemble_all_stations.csv
  Rows : 11,760

Preview — first station, all months of 1995:
Station_ID   Station_Name   Basin  Year  Month  pr_mm_month
    AB0004 KH.EL-WAHADNEH N.R.S.W  1995      1      90.9484
    AB0004 KH.EL-WAHADNEH N.R.S.W  1995      2      69.4518
    AB0004 KH.EL-WAHADNEH N.R.S.W  1995      3      43.1631
    AB0004 KH.EL-WAHADNEH N.R.S.W  1995      4      25.2334
    AB0004 KH.EL-WAHADNEH N.R.S.W  1995      5       2.4564
    AB0004 KH.EL-WAHADNEH N.R.S.W  1995      6       0.0000
    AB0004 KH.EL-WAHADNEH N.R.S.W  1995      7       0.0000
    AB0004 KH.EL-WAHADNEH N.R.S.W  1995      8       0.0000
    AB0004 KH.EL-WAHADNEH N.R.S.W  1995      9       0.0007
    AB0004 KH.EL-WAHADNEH N.R.S.W  1995     10       3.1065
    AB0004 KH.EL-WAHADNEH N.R.S.W  1995     11      20.5824
    AB0004 KH.EL-WAHADNEH N.R.S.W  1995     12     132.3244


## 7. Compute Long-Term Monthly Means (Ensemble Climatology)

Monthly totals are averaged across all years in the evaluation period for each calendar month

$$\text{LTMA}(m) = \frac{1}{Y}\sum_{y=1}^{Y}\text{MS}(y, m)$$

where $Y = 20$ years (1995–2014).

Output: `long_term_monthly_mean/ltmm_ensemble_all_stations.csv`  
Columns: `Station_ID`, `Station_Name`, `Basin`, `Month`, `Month_Name`, `LTMM_mm_month`, `N_Years`


In [7]:
ltmm_csv = LTMM_OUT / "ltmm_ensemble_all_stations.csv"

if ltmm_csv.exists():
    print(f"[SKIP] LTMM CSV already exists: {ltmm_csv}")
    ltmm_df = pd.read_csv(ltmm_csv)
else:
    ltmm_df = (
        monthly_df
        .groupby(["Station_ID","Station_Name","Basin","Month"], sort=True)
        ["pr_mm_month"]
        .agg(LTMM_mm_month="mean", N_Years="count")
        .reset_index()
    )
    ltmm_df["LTMM_mm_month"] = ltmm_df["LTMM_mm_month"].round(4)
    ltmm_df["Month_Name"]    = ltmm_df["Month"].map(MONTH_NAMES)
    ltmm_df["Model"]         = "3-Model Ensemble"

    ltmm_df = ltmm_df[["Model","Station_ID","Station_Name","Basin",
                         "Month","Month_Name","LTMM_mm_month","N_Years"]]
    ltmm_df.to_csv(ltmm_csv, index=False)
    print(f"LTMM CSV saved: {ltmm_csv}")

print(f"  Rows : {len(ltmm_df):,}  ({ltmm_df['Station_ID'].nunique()} stations × 12 months)")
print()
print("Preview — first station (12 months):")
sid0 = ltmm_df["Station_ID"].iloc[0]
print(ltmm_df[ltmm_df["Station_ID"] == sid0]
      [["Station_ID","Station_Name","Basin","Month_Name","LTMM_mm_month","N_Years"]]
      .to_string(index=False))


LTMM CSV saved: C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\ensemble data\long_term_monthly_mean\ltmm_ensemble_all_stations.csv
  Rows : 588  (49 stations × 12 months)

Preview — first station (12 months):
Station_ID   Station_Name   Basin Month_Name  LTMM_mm_month  N_Years
    AB0004 KH.EL-WAHADNEH N.R.S.W    January        90.5922       20
    AB0004 KH.EL-WAHADNEH N.R.S.W   February        81.5423       20
    AB0004 KH.EL-WAHADNEH N.R.S.W      March        64.6073       20
    AB0004 KH.EL-WAHADNEH N.R.S.W      April        21.9108       20
    AB0004 KH.EL-WAHADNEH N.R.S.W        May         3.6780       20
    AB0004 KH.EL-WAHADNEH N.R.S.W       June         0.1292       20
    AB0004 KH.EL-WAHADNEH N.R.S.W       July         0.0000       20
    AB0004 KH.EL-WAHADNEH N.R.S.W     August         0.0000       20
    AB0004 KH.EL-WAHADNEH N.R.S.W  September         0.0615       20
    AB0004 KH.EL-WAHADNEH N.R.S.W    October        12.0064       20
    AB0004 KH.EL-WAHADNEH 

## 8. Output Summary

In [8]:
ds_ens.close()

print("=" * 65)
print("OUTPUT DIRECTORY STRUCTURE")
print("=" * 65)
for root, dirs, files in os.walk(OUTPUT_ROOT):
    dirs[:] = sorted(d for d in dirs if not d.startswith("."))
    level   = Path(root).relative_to(OUTPUT_ROOT).parts
    indent  = "  " * len(level)
    print(f"{indent}📁 {Path(root).name}/")
    sub = "  " * (len(level) + 1)
    for f in sorted(files):
        sz = (Path(root) / f).stat().st_size / 1024
        unit = "KB"
        if sz > 1024:
            sz /= 1024; unit = "MB"
        print(f"{sub}📄 {f}  ({sz:.1f} {unit})")

print()
print("Next step → run Notebook 07_Ensemble_Validation.ipynb")


OUTPUT DIRECTORY STRUCTURE
📁 ensemble data/
  📁 daily/
    📄 daily_ensemble_all_stations.csv  (17.0 MB)
  📁 long_term_monthly_mean/
    📄 ltmm_ensemble_all_stations.csv  (40.2 KB)
  📁 monthly/
    📄 monthly_ensemble_all_stations.csv  (545.4 KB)

Next step → run Notebook 07_Ensemble_Validation.ipynb
